# Visualizzazione di onde sonore e intensità in funzione della frequenza
* Per onde puntiformi, puntiformi in movimento verso l'osservatore e rumore casuale.
* Dovremo usare dei concetti avanzati di programmazione in Python
  * [Ereditarietà e polimorfismo](10_08.ipynb)
  * [Decoratori](10_09.ipynb)

In [1]:
import ROOT
import numpy as np
import random
sound_speed = 340. # m/s

In [2]:
class soundSource(object) :
    """classe GENERICA per una sorgente di suoni"""
    
    def __init__(self,amplitude) :
        self._amplitude = amplitude

    def __str__(self) :
        return f'Source data : \n Amplitude = {self._amplitude:.2f} a.u.'

    def wave_vs_time(self,t) :
        raise NotImplementedError

    def intensity_vs_frequency(self,nu,fwhm,t) :
        raise NotImplementedError

In [3]:
class whiteNoise(soundSource) :
    """classe per rumore bianco"""

    def __str__(self) :
        return (super().__str__() + '\n Type = white noise')

    def wave_vs_time(self,t) :
        if (isinstance(t,float)) :
            return self._amplitude*2.*(random.uniform(0.,1.) - 0.5)
        return self._amplitude*2.*(np.random.rand(t.shape[0]) - 0.5)
        # 2*(random - 1/2) dà un numero casuale tra -1 e 1
        # t.shape[0] è la lunghezza dell'array t ("shape" è una tupla)

    def intensity_vs_frequency(self,nu,fwhm,t) :
        return self.wave_vs_time(t)**2

In [4]:
class pointSource(soundSource) :
    """classe per sorgente di suoni puntiforme"""
    
    def __init__(self,amplitude,frequency,phase,position) :
        if (frequency < 20 or frequency > 20000) :
             raise ValueError('Sound frequency must be between 20 and 20000 Hz')
        super().__init__(amplitude)
        self._frequency = frequency
        self._phase = phase
        self._position = position

    def __str__(self) :
        return (super().__str__() + '\n Type = Point source'
               + f'\n Frequency = {self._frequency:.2f} Hz'
               + f'\n Phase = {self._phase:.2f} rad'
               + f'\n Position = [{self._position.X():.2f}, {self._position.Y():.2f}, {self._position.Z():.2f}] m')

    def current_r(self,t) :
        current_x = self._position.X()
        current_y = self._position.Y()
        current_z = self._position.Z()
        return np.sqrt(np.power(current_x,2) + np.power(current_y,2) + np.power(current_z,2))
        
    def wave_vs_time(self,t) :
        return self._amplitude/self.current_r(t)*np.sin(2*np.pi*self._frequency*t + self._phase) 

    def intensity_vs_frequency(self,nu,fwhm,t) :
        full_intensity = self.wave_vs_time(t)**2
        sigma = fwhm/2.355
        return full_intensity*np.exp(-np.power((self._frequency-nu)/sigma, 2.)/2)

In [5]:
class movingPointSource(pointSource) :
    """classe per sorgente di suoni puntiforme in movimento verso o in direzione opposta all'osservatore"""
    
    def __init__(self,amplitude,frequency,phase,position,speed) :
        super().__init__(amplitude,frequency,phase,position)
        self._speed = speed
        self._frequency *= sound_speed/(sound_speed-self._speed)    # effetto Doppler
        
    def __str__(self) :
        return (super().__str__() 
               + f'\n Velocity = {self._speed:.2f} m/s')

    def current_r(self,t) :
        current_x = self._position.X() - self._speed * self._position.X()/self._position.Mag() * t
        current_y = self._position.Y() - self._speed * self._position.Y()/self._position.Mag() * t
        current_z = self._position.Z() - self._speed * self._position.Z()/self._position.Mag() * t
        return np.sqrt(np.power(current_x,2) + np.power(current_y,2) + np.power(current_z,2))

In [6]:
position_mps = ROOT.TVector3(250.,0.,0.)
mps = movingPointSource(500.,25.,0.,position_mps,8.)

In [7]:
print (mps)

Source data : 
 Amplitude = 500.00 a.u.
 Type = Point source
 Frequency = 25.60 Hz
 Phase = 0.00 rad
 Position = [250.00, 0.00, 0.00] m
 Velocity = 8.00 m/s


In [8]:
c1 = ROOT.TCanvas("c1","c1",1400,400)

In [9]:
t_min, t_max, n_points = 0.,200.,400    # VISUALIZZA DOPPLER
t = np.linspace(t_min,t_max,n_points)

In [10]:
wave = mps.wave_vs_time(t)

In [11]:
gr = ROOT.TGraph(n_points,t,wave)
gr.Draw("ALP")
c1.Draw()

In [12]:
def total_wave_vs_time(wave_list,t) :
    return sum([wave.wave_vs_time(t) for wave in wave_list])

In [13]:
wn = whiteNoise(0.2)

In [14]:
list_waves = list([wn,mps])

In [15]:
t_min, t_max, n_points = 0.,0.3,500     # VISUALIZZA RUMORE
t = np.linspace(t_min,t_max,n_points)
wave = mps.wave_vs_time(t)
wave1 = total_wave_vs_time(list_waves,t)

In [16]:
gr = ROOT.TGraph(n_points,t,wave)
gr1 = ROOT.TGraph(n_points,t,wave1)
gr1.SetLineColor(ROOT.kRed)
gr.Draw("ALP")
gr1.Draw("LP SAME")
c1.Update()

In [17]:
c1.Draw()

In [18]:
sps = pointSource(500.,25.,0.,position_mps)
sps2 = pointSource(500.,24.,0.,position_mps)
sps3 = pointSource(250.,24.,np.pi,position_mps)
sps4 = pointSource(500.,24.,np.pi,position_mps)

In [19]:
list_waves = list([sps,sps2,wn])

In [20]:
t_min, t_max, n_points = 0.,2.,1000     # VISUALIZZA BATTIMENTI
t = np.linspace(t_min,t_max,n_points)    
wave2 = total_wave_vs_time(list_waves,t)

In [21]:
gr3 = ROOT.TGraph(n_points,t,wave2)
gr3.Draw("ALP")
c1.Update()

In [22]:
c1.Draw()

In [23]:
from collections import defaultdict

def total_intensity_vs_frequency(wave_list,nu,fwhm,t) :
    total_intensity = 0.

    # first group in coherent/incoherent
    list_incoherent = []
    group_coherent = defaultdict(list)
    for wave in wave_list:
       if (isinstance(wave,pointSource)) : 
           group_coherent[wave._frequency].append(wave)
       else :
           list_incoherent.append(wave)

    # simply add intensities if are intrinsically incoherent (noise), or there is just one source of that frequency
    for wave_inc in list_incoherent :
        total_intensity += wave_inc.intensity_vs_frequency(nu,fwhm,t)
    for wave_coh in group_coherent.values() :
        if (len(wave_coh) == 1) :
            total_intensity += wave_coh[0].intensity_vs_frequency(nu,fwhm,t)
        elif (len(wave_coh) == 2) : 
            # use interference rules
            i1 = wave_coh[0].intensity_vs_frequency(nu,fwhm,t)
            i2 = wave_coh[1].intensity_vs_frequency(nu,fwhm,t)
            phi1 = wave_coh[0]._phase
            phi2 = wave_coh[1]._phase
            relative_phase = phi2 - phi1 + 2*np.pi*wave_coh[0]._frequency/sound_speed*( wave_coh[1].current_r(t)-wave_coh[0].current_r(t) ) 
            total_intensity += i1 + i2 + 2*np.sqrt(i1*i2)*np.cos(relative_phase)
        else:
            raise NotImplementedError('Interference of more than 2 sources is not implemented')
    return total_intensity

In [24]:
nu_min, nu_max, n_points = 20.,30.,400
nu1 = np.linspace(nu_min,nu_max,n_points) 
int1 = sps3.intensity_vs_frequency(nu1,0.1,0.234)

In [25]:
c2 = ROOT.TCanvas("c2","c2",1400,400)

In [26]:
gr4 = ROOT.TGraph(n_points,nu1,int1)
gr4.Draw("ALP")
c2.Draw()

In [27]:
list_waves = list([sps,sps2,sps4,wn])

In [28]:
int2 = total_intensity_vs_frequency(list_waves,nu1,0.1,0.234)

In [29]:
gr4 = ROOT.TGraph(n_points,nu1,int2)
gr4.Draw("ALP")
c2.Update()

In [30]:
c3 = ROOT.TCanvas("c3","c3",1400,400)

In [31]:
def drawWave(gr_wave,c) :
    c.cd()
    gr_wave.SetTitle("Sound wave vs. time")
    gr_wave.GetXaxis().SetTitle("Time [s]")
    gr_wave.GetYaxis().SetTitle("Sound amplitude [a.u.]")
    gr_wave.Draw()
    c.Update()

In [32]:
drawWave(gr3,c3)

In [33]:
def drawIntensity(gr_int,c) :
    c.cd()
    gr_int.SetTitle("Sound intensity vs. frequency")
    gr_int.GetXaxis().SetTitle("Frequency [Hz]")
    gr_int.GetYaxis().SetTitle("Sound intensity [a.u.]")
    gr_int.Draw()
    c.Update()

In [34]:
c4 = ROOT.TCanvas("c4","c4",1400,400)
drawIntensity(gr4,c4)

In [35]:
def graph_decorator(func):
    def wrapper(gr,c):
        c.SetBottomMargin(0.16)
        gr.SetLineColor(ROOT.kRed)
        gr.SetLineWidth(2)
        gr.GetXaxis().SetTitleSize(0.06)
        gr.GetXaxis().SetTitleOffset(1.3)
        gr.GetXaxis().SetLabelSize(0.06)
        gr.GetYaxis().SetTitleSize(0.06)
        gr.GetYaxis().SetTitleOffset(0.7)
        gr.GetYaxis().SetLabelSize(0.06)
        result = func(gr,c)
        return result
    return wrapper

@graph_decorator
def drawWave(gr_wave,c) :
    c.cd()
    gr_wave.SetTitle("Sound wave vs. time")
    gr_wave.GetXaxis().SetTitle("Time [s]")
    gr_wave.GetYaxis().SetTitle("Sound amplitude [a.u.]")
    gr_wave.Draw()
    c.Update() 

In [36]:
drawWave(gr3,c3)

In [37]:
c3.Draw()

In [38]:
@graph_decorator
def drawIntensity(gr_int,c) :
    c.cd()
    gr_int.SetTitle("Sound intensity vs. frequency")
    gr_int.GetXaxis().SetTitle("Frequency [Hz]")
    gr_int.GetYaxis().SetTitle("Sound intensity [a.u.]")
    gr_int.Draw()
    c.Update()

In [39]:
drawIntensity(gr4,c4)

In [40]:
c4.Draw()